In [9]:
using StatsBase, Random, Distributions, Plots, LinearAlgebra

# Differential evolution

In [10]:

function differential_evolution(f, population, k_max; p=0.5, w=1)
    n, m = length(population[1]), length(population)
    for k in 1:k_max
        for (k, x) in enumerate(population)
            a, b, c = sample(population,
                Weights([j != k for j in 1:m]), 3, replace=false)
            z = a + w * (b - c)
            j = rand(1:n)
            x′ = [i == j || rand() < p ? z[i] : x[i] for i in 1:n]
            if f(x′) < f(x)
                x[:] = x′
            end

        end
        @show f(population[argmin(f.(population))])
    end
    return population[argmin(f.(population))]
end

differential_evolution (generic function with 1 method)

In [11]:
function rand_population_uniform(m, a, b)
    d = length(a)
    return [a + rand(d) .* (b - a) for i in 1:m]
end

function rand_population_normal(m, μ, Σ)
    D = MvNormal(μ, Σ)
    return [rand(D) for i in 1:m]

end

function rand_population_cauchy(m, μ, σ)
    n = length(μ)
    return [[rand(Cauchy(μ[j], σ[j])) for j in 1:n] for i in 1:m]
end

rand_population_binary(m, n) = [bitrand(n) for i in 1:m]

rand_population_binary (generic function with 1 method)

In [12]:
f = x -> norm(x)
m = 100 # population size
k_max = 100 # number of iterations
population = rand_population_uniform(m, [-300, -300], [300, 300])
best = differential_evolution(f, population,k_max)

f(population[argmin(f.(population))]) = 9.873166470520664
f(population[argmin(f.(population))]) = 9.873166470520664
f(population[argmin(f.(population))]) = 9.873166470520664
f(population[argmin(f.(population))]) = 9.873166470520664
f(population[argmin(f.(population))]) = 7.1703080025153785
f(population[argmin(f.(population))]) = 7.1703080025153785
f(population[argmin(f.(population))]) = 4.133817638814007
f(population[argmin(f.(population))]) = 4.133817638814007
f(population[argmin(f.(population))]) = 4.133817638814007
f(population[argmin(f.(population))]) = 4.133817638814007
f(population[argmin(f.(population))]) = 4.133817638814007
f(population[argmin(f.(population))]) = 4.133817638814007
f(population[argmin(f.(population))]) = 4.133817638814007
f(population[argmin(f.(population))]) = 4.133817638814007
f(population[argmin(f.(population))]) = 4.133817638814007
f(population[argmin(f.(population))]) = 4.1237502235913
f(population[argmin(f.(population))]) = 1.6628380030527268
f(population[

2-element Vector{Float64}:
 4.018398271909973e-7
 1.057411139981923e-6

# Particle Swarm optimization

In [13]:
mutable struct Particle
    x
    v
    x_best

end

In [14]:
function rand_population_uniform_particles(m, a, b)
    d = length(a)
    population = [Particle(a + rand(d) .* (b - a), zeros(d), zeros(d)) for i in 1:m]
    for p in population
        p.x_best = p.x
    end
    return population
   
end

rand_population_uniform_particles (generic function with 1 method)

In [15]:
function particle_swarm_optimization(f, population, k_max; w=1, c1=1, c2=1)
    n = length(population[1].x)
    x_best, y_best = copy(population[1].x_best), Inf
    for P in population
        y = f(P.x)
        if y < y_best
            x_best[:], y_best = P.x, y
        end
    end
    for k in 1:k_max
        for P in population
            r1, r2 = rand(n), rand(n)
            P.x += P.v
            P.v = w * P.v + c1 * r1.*(P.x_best - P.x) +
                            c2 * r2.*(x_best - P.x)
            y = f(P.x)
            if y < y_best
                x_best[:], y_best = P.x, y
            end
            if y < f(P.x_best)
                P.x_best[:] = P.x
            end

        end
    end
    return y_best, x_best
end

particle_swarm_optimization (generic function with 1 method)

In [16]:
f = x -> norm(x)
m = 100 # population size
k_max = 500 # number of iterations
population = rand_population_uniform_particles(m, [-300, -300,-300,-300], [300, 300,300,300])
best = particle_swarm_optimization(f, population,k_max; w= 0.5)

(6.910506555413219e-85, [3.713760704282409e-85, -8.805988316265032e-86, -4.128740006283398e-85, 4.0176030095057795e-85])

# TODO

- test DE and PSO on Ackley's function with 5 and 10 dimensions:
  -  find suitable parameters
  -  perform multiple runs
  -  compute mean and standard deviation of results. 